In [1]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the data exported from Week 5.
regression_df = pd.read_csv(
    "regression_data.csv",
    parse_dates=["hour_local"]
)

# All remaining columns are predictors.
core_features = regression_df.columns.drop(
    ["hour_local", "avg_departure_delay"]
).tolist()

print(f"Loaded {len(regression_df):,} rows and {len(core_features)} predictors.")

Loaded 28,499 rows and 23 predictors.


In [2]:
# Sort our data by date and time
data = regression_df.sort_values("hour_local")

# X contains the features we use to make predictions
X = data[core_features]

# y contains the variable we want to predict
y = data["avg_departure_delay"]

In [3]:
# Sort data chronologically
data = regression_df.sort_values("hour_local")

X = data[core_features]
y = data["avg_departure_delay"]

In [4]:
from sklearn.model_selection import train_test_split
#Split into training, validation, and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42

Below is the implementation of quantile regressors where the quantiles are 5, 10, 50, 90, and 95. These values were chosen because quantile regression is often best for the upper and lower ends of the data, where more extreme values might not be captured by OLS. The 50th quantile is chosen as a benchmark to compare to linear regression.

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import genpareto
from sklearn.linear_model import LinearRegression, QuantileRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_pinball_loss

#Fit some models
q05 = QuantileRegressor(quantile=0.05, alpha=0).fit(X_train, y_train)
q10 = QuantileRegressor(quantile=0.10, alpha=0).fit(X_train, y_train)
q50 = QuantileRegressor(quantile=0.50, alpha=0).fit(X_train, y_train)
q90 = QuantileRegressor(quantile=0.90, alpha=0).fit(X_train, y_train)
q95 = QuantileRegressor(quantile=0.95, alpha=0).fit(X_train, y_train)

#Have these models attempt to predict from validation data
pred_q05 = q05.predict(X_test)
pred_q10 = q10.predict(X_test)
pred_q50 = q50.predict(X_test)
pred_q90 = q90.predict(X_test)
pred_q95 = q95.predict(X_test)

print("Median Quantile MAE:", mean_absolute_error(y_test, pred_q50))

print("Q05 Pinball Loss:",
      mean_pinball_loss(y_test, pred_q05, alpha=0.05))
print("Q10 Pinball Loss:",
      mean_pinball_loss(y_test, pred_q10, alpha=0.10))
print("Q50 Pinball Loss:",
      mean_pinball_loss(y_test, pred_q50, alpha=0.50))
print("Q90 Pinball Loss:",
      mean_pinball_loss(y_test, pred_q90, alpha=0.90))
print("Q95 Pinball Loss:",
      mean_pinball_loss(y_test, pred_q50, alpha=0.95))

print("Q05 Coverage:", np.mean(y_test <= pred_q05))
print("Q10 Coverage:", np.mean(y_test <= pred_q10))
print("Q50 Coverage:", np.mean(y_test <= pred_q50))
print("Q90 Coverage:", np.mean(y_test <= pred_q90))
print("Q95 Coverage:", np.mean(y_test <= pred_q95))

Median Quantile MAE: 12.428068043629322
Q05 Pinball Loss: 0.9532713675108955
Q10 Pinball Loss: 1.7805763953388014
Q50 Pinball Loss: 6.214034021814661
Q90 Pinball Loss: 5.0147105969303
Q95 Pinball Loss: 8.791838505034953
Q05 Coverage: 0.056842105263157895
Q10 Coverage: 0.10970760233918128
Q50 Coverage: 0.5188304093567252
Q90 Coverage: 0.896608187134503
Q95 Coverage: 0.9499415204678363


As can be seen by the given metrics, the quantile regression performs quite well! The pinball losses for Q05 and Q10 are particularly low, and Q95 has the highest pinball loss. The coverage of the models is also extremely close to the coverage each model should theoretically cover. The differences between the coverage and the model's quantile are as follows:
- Q05: 0.00684 (0.684%)
- Q10: 0.00971 (0.971%)
- Q50: 0.01883 (1.883%)
- Q90: -0.00339 (-0.339%)
- Q95: 0.00006 (0.006%)

The quantile regressor also performs with a mean absolute error of 12.428, which is better than the MAE of the ridge regression model we found before that used 21 features and had a MAE of 13.650. This means that predicting the conditional median is producing more accurate predictions (in terms of MAE) than predicting the conditional mean.